In [6]:
import heapq
from collections import deque

graph = {"A": ["B", "C"], "B": ["D", "E"], "C": ["F"],
         "D": [], "E": ["G"], "F": [], "G": []}

def bfs(start, goal, g=None):
    g = g or graph
    q = deque([(start, [start])])
    seen = {start}
    while q:
        node, path = q.popleft()
        if node == goal:
            return path
        for n in g[node]:
            if n not in seen:
                seen.add(n)
                q.append((n, path + [n]))

def dfs(start, goal, g=None):
    g = g or graph
    stack = [(start, [start])]
    seen = set()
    while stack:
        node, path = stack.pop()
        if node == goal:
            return path
        if node in seen:
            continue
        seen.add(node)
        for n in reversed(g[node]):
            stack.append((n, path + [n]))

def bfs_trace(start, goal, g=None):
    g = g or graph
    q = deque([(start, [start])])
    seen = {start}
    visited, rows = [], []
    while q:
        node, path = q.popleft()
        visited.append(node)
        if node == goal:
            rows.append((len(visited), node, [p[0] for p in q], path, "GOAL"))
            return path, visited, rows
        for n in g[node]:
            if n not in seen:
                seen.add(n)
                q.append((n, path + [n]))
        rows.append((len(visited), node, [p[0] for p in q], path, ""))
    return None, visited, rows


def dfs_trace(start, goal, g=None):
    g = g or graph
    stack = [(start, [start])]
    seen = set()
    visited, rows = [], []
    while stack:
        node, path = stack.pop()
        if node == goal:
            visited.append(node)
            rows.append((len(visited), node, [p[0] for p in stack], path, "GOAL"))
            return path, visited, rows
        if node in seen:
            continue
        seen.add(node)
        visited.append(node)
        for n in reversed(g[node]):
            stack.append((n, path + [n]))
        rows.append((len(visited), node, [p[0] for p in stack], path, ""))
    return None, visited, rows


FOUND, CUTOFF, FAILURE = "FOUND", "CUTOFF", "FAILURE"


def dls(node, goal, limit, g=None, path=None, counter=None):
    """Depth-limited search. Returns (status, path).
    CUTOFF  = the limit stopped us while a deeper continuation exists.
    FAILURE = this branch is exhausted (nothing deeper to explore).
    Cycle check is path-local: only nodes on the current path are skipped."""
    g = g if g is not None else graph
    path = path or [node]
    if counter is not None:
        counter[0] += 1
    if node == goal:
        return FOUND, path
    if limit == 0:
        deeper = any(n not in path for n in g[node])
        return (CUTOFF, None) if deeper else (FAILURE, None)
    cutoff = False
    for n in g[node]:
        if n in path:
            continue
        status, result = dls(n, goal, limit - 1, g, path + [n], counter)
        if status == FOUND:
            return FOUND, result
        if status == CUTOFF:
            cutoff = True
    return (CUTOFF, None) if cutoff else (FAILURE, None)


def iddfs(start, goal, max_depth, g=None):
    """Repeat DLS with limits 0..max_depth, using a fresh search each time.
    Returns (status, path, attempts) where attempts = [(limit, status, nodes)]."""
    attempts = []
    for limit in range(max_depth + 1):
        counter = [0]
        status, path = dls(start, goal, limit, g, None, counter)
        attempts.append((limit, status, counter[0]))
        if status == FOUND:
            return FOUND, path, attempts
        if status == FAILURE:
            return FAILURE, None, attempts
    return CUTOFF, None, attempts


wgraph = {"A": [("B", 1), ("C", 5)],
          "B": [("D", 2), ("E", 1)],
          "C": [("F", 3), ("G", 7)],
          "D": [], "E": [("G", 1)], "F": [], "G": []}


def ucs(start, goal, g=None):
    """Uniform-cost search. Returns (path, cost, expanded_order).
    Assumes nonnegative edge costs."""
    g = g or wgraph
    tie = 0
    frontier = [(0, tie, start, [start])]
    best = {}
    expanded = []
    while frontier:
        cost, _, node, path = heapq.heappop(frontier)
        if node in best and best[node] <= cost:
            continue
        best[node] = cost
        expanded.append(node)
        if node == goal:
            return path, cost, expanded
        for n, w in g[node]:
            tie += 1
            heapq.heappush(frontier, (cost + w, tie, n, path + [n]))
    return None, None, expanded


def path_cost(path, g=None):
    g = g or wgraph
    total = 0
    for a, b in zip(path, path[1:]):
        total += dict(g[a])[b]
    return total


def plain(g):
    """Drop weights to get an unweighted successor graph for BFS."""
    return {k: [n for n, _ in v] for k, v in g.items()}


# ---------------------------------------------------------------
# Printing helpers
# ---------------------------------------------------------------
LINE = "-" * 78


def print_trace(title, rows, key):
    print("\n" + title)
    print(LINE)
    print(f"{'Step':<5}{'Processed':<11}{key + ' after step':<22}{'Path'}")
    print(LINE)
    for step, node, frontier, path, tag in rows:
        f = "[" + ", ".join(frontier) + "]"
        print(f"{step:<5}{node:<11}{f:<22}{' -> '.join(path)} {tag}")


if __name__ == "__main__":
    # ===== TASK 1: manual-trace verification =====
    print("TASK 1: BFS and DFS on the original graph (start A, goal G)")
    b_path, b_visit, b_rows = bfs_trace("A", "G")
    d_path, d_visit, d_rows = dfs_trace("A", "G")
    print_trace("BFS trace (queue, front on the left)", b_rows, "Queue")
    print_trace("DFS trace (stack, top on the right)", d_rows, "Stack")
    print("\nBFS final path :", b_path, "| visit order:", b_visit,
          f"({len(b_visit)} nodes)")
    print("DFS final path :", d_path, "| visit order:", d_visit,
          f"({len(d_visit)} nodes)")
    print("Matches supplied bfs():", b_path == bfs("A", "G"))
    print("Matches supplied dfs():", d_path == dfs("A", "G"))

    # ===== TASK 2: DLS and IDDFS =====
    print("\n\nTASK 2: Depth-limited search, limits 1-4")
    print(LINE)
    print(f"{'Limit':<7}{'Status':<10}{'Nodes processed':<17}{'Path'}")
    print(LINE)
    for limit in [1, 2, 3, 4]:
        c = [0]
        status, path = dls("A", "G", limit, None, None, c)
        print(f"{limit:<7}{status:<10}{c[0]:<17}{path}")

    print("\nIDDFS with maximum depths 1-4")
    print(LINE)
    for md in [1, 2, 3, 4]:
        status, path, attempts = iddfs("A", "G", md)
        limits = [a[0] for a in attempts]
        work = sum(a[2] for a in attempts)
        print(f"max_depth={md}: {status:<8} path={path} limits tried={limits} "
              f"total nodes processed={work}")
    print("\nIDDFS (max 4) work per limit: limit, status, nodes processed")
    for a in iddfs("A", "G", 4)[2]:
        print("  ", a)

    # ===== EXTENSION: weighted corridors =====
    print("\n\nEXTENSION: weighted corridors with added C -> G")
    uw = plain(wgraph)
    bp = bfs("A", "G", uw)
    up, uc, uexp = ucs("A", "G")
    print(f"BFS : path={bp} hops={len(bp) - 1} cost={path_cost(bp)}")
    print(f"UCS : path={up} hops={len(up) - 1} cost={uc} "
          f"expanded order={uexp}")

    # ===== EDGE CASES =====
    print("\n\nEDGE CASES")
    print(LINE)
    print("1) Start equals goal (A -> A)")
    print("   BFS:", bfs("A", "A"), "| DFS:", dfs("A", "A"),
          "| DLS limit 0:", dls("A", "A", 0)[0],
          "| UCS:", ucs("A", "A")[:2])

    print("2) Unreachable goal")
    print("   C -> G on original graph: BFS:", bfs("C", "G"),
          "| DFS:", dfs("C", "G"))
    print("   DLS C->G limit 1:", dls("C", "G", 1)[0],
          "| limit 3:", dls("C", "G", 3)[0])
    s, p, att = iddfs("C", "G", 4)
    print("   IDDFS C->G max 4:", s, "| attempts:", att)
    print("   UCS D -> G (extended graph):", ucs("D", "G")[:2])

    cyc = {"A": ["B"], "B": ["C"], "C": ["A", "D"], "D": []}
    print("3) Graph with a cycle (A -> B -> C -> A, exit D)")
    print("   BFS:", bfs("A", "D", cyc), "| DFS:", dfs("A", "D", cyc))
    print("   DLS limit 3:", dls("A", "D", 3, cyc))
    print("   IDDFS max 6:", iddfs("A", "D", 6, cyc))
    print("   Unreachable goal Z in cyclic graph, DLS limit 2:",
          dls("A", "Z", 2, cyc)[0], "| limit 3:", dls("A", "Z", 3, cyc)[0])
    print("   IDDFS A->Z max 6:", iddfs("A", "Z", 6, cyc))

    print("4) Bounded search is not proof of unreachability")
    print("   DLS A->G limit 2 gives", dls("A", "G", 2)[0],
          "but G is reachable at depth 3.")

TASK 1: BFS and DFS on the original graph (start A, goal G)

BFS trace (queue, front on the left)
------------------------------------------------------------------------------
Step Processed  Queue after step      Path
------------------------------------------------------------------------------
1    A          [B, C]                A 
2    B          [C, D, E]             A -> B 
3    C          [D, E, F]             A -> C 
4    D          [E, F]                A -> B -> D 
5    E          [F, G]                A -> B -> E 
6    F          [G]                   A -> C -> F 
7    G          []                    A -> B -> E -> G GOAL

DFS trace (stack, top on the right)
------------------------------------------------------------------------------
Step Processed  Stack after step      Path
------------------------------------------------------------------------------
1    A          [C, B]                A 
2    B          [C, E, D]             A -> B 
3    D          [C, E]        